> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

# 00. 작은 모델 하나를 훈련하고 결과 판단하기 — 120분

Python 리스트·함수 이후 처음 배우는 훈련 실습입니다. 대표 모델은 LSTM이며 이 작은 숫자 기억 과제는 자연어 LLM이나 작업대 인식 모델 자체가 아닙니다.

| 시간 | 할 일 |
|---|---|
| 0–15분 | 입력·정답·훈련/검증 구분, 텐서 모양 읽기 |
| 15–35분 | 준비된 loss 틀 읽기·TODO·오류 복구 |
| 35–60분 | LSTM quick 훈련 1회·훈련 전후 비교 |
| 60–80분 | 저장한 가중치를 새 프로세스로 읽어 추론 |
| 80–95분 | 같은 과제의 RNN/LSTM 참고 결과 읽기·구조 시연 |
| 95–110분 | 도입 판단: 데이터·기존 모델·직접 훈련 선택 |
| 110–120분 | 저장·질문·재실행 복구 |

입력→예측→정답 비교→파라미터 수정이 반복되는 것을 관찰합니다. gate 수식·attention·causal mask 직접 구현은 01–04 자습 노트북에 보존했습니다. 훈련이 빨리 끝나면 다음 모델을 추가하지 말고 틀린 예측을 읽습니다.

<!-- standalone-review:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** Python 3 커널에서 첫 환경 셀의 `/opt/course/bin/python` 경로를 확인합니다. 코드 셀은 Shift+Enter로 실행하고 `[*]`가 끝난 뒤 다음 셀로 이동합니다. TODO 파일을 저장한 뒤 검증 셀을 실행합니다.

### 현재 실습의 입력·저장·실행 범위

열 파일은 `/workspace/04.LLM/00_training_core.ipynb`입니다. 첫 환경 셀은 `ROOT`, `COURSE`, `DATA`, `OUTPUTS`, `check`, `load_op`, `run_training`을 **이 커널에 다시 정의** 합니다. Python 경로와 PyTorch·CUDA 출력에서 CUDA가 `True`인지 확인하고 같은 Jetson의 다른 사용자와 GPU 실행 순서를 맞춥니다. 이 숫자 모델 훈련에는 Ollama 서버가 필요하지 않습니다.

입력은 `/workspace/Data/datasets/04.LLM/digits.json`과 `.sha256`입니다. 이 데이터 640개를 train 512개·validation 128개로 나누며, PAD/BOS/EOS=0/1/2, 숫자 0~9의 token ID=3~12입니다. 파일이 없으면 배포 데이터가 포함된 개인 자료 폴더인지 확인합니다. 이전 훈련 checkpoint 없이 새 훈련을 시작합니다.

`exercises/*.py`는 Jupyter 파일 편집기에서 수정·저장하세요. 노트북의 파일 출력 셀은 읽기 전용이고 `check(..., 'exercise')`는 저장한 최신 파일을 읽습니다. 결과는 `/workspace/Data/outputs/04.LLM/<실행명>/metrics.json`·`loss.csv`, 자신의 가중치는 `/workspace/Data/checkpoints/users/04.LLM/`에 남습니다. `run_training`은 새 결과 폴더를 만들고 반환한 `report['checkpoint']`는 `Data` 기준 상대 경로입니다. 다른 노트북의 `report` 변수는 이어지지 않습니다.

<!-- standalone-review:end -->


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

훈련은 정답을 외부에서 알려 주면서 예측이 틀린 정도를 줄이도록 모델의 가중치를 바꾸는 과정입니다. 작은 숫자 기억 문제를 사용하면 입력·정답·loss·업데이트를 직접 추적할 수 있습니다. 여기서 만든 LSTM이 오후에 사용하는 Gemma로 변하는 것은 아닙니다.

### 전체 과정에서 내 코드가 하는 일

```text
훈련·검증 데이터 분리
  ↓
예측과 정답의 위치 정렬
  ↓
loss TODO
  ↓
역전파·가중치 갱신
  ↓
전후 검증
  ↓
체크포인트 재로드
```

loss에 전달할 logits와 target의 모양을 맞춥니다. 훈련 실행기는 제공되며 학생은 손실 계산과 실제 지표 해석을 담당합니다. 훈련에 사용하지 않은 검증 입력과 새 프로세스 재로드로 결과를 확인합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 훈련 loss가 작아졌는데 전체 정답 일치율이 낮을 수 있나요?

**설명:** 가능합니다. token 하나씩 맞힌 비율과 출력 열 전체가 맞은 비율은 다릅니다. 검증 지표·틀린 예측·저장 후 재로드를 함께 봐야 하며 한 숫자로 일반적인 기억 능력이나 업무 성능을 보장하지 않습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **준비 셀과 「1. 숫자 다섯 개를 기억하는 작은 과제」** → [2일차 슬라이드 10쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s005) · [배포 PDF 10쪽](../슬라이드/2일차.pdf#page=10)에서 17쪽까지 복습합니다.
- **「수업 핵심 TODO — loss」** → [2일차 슬라이드 18쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s013) · [배포 PDF 18쪽](../슬라이드/2일차.pdf#page=18)에서 22쪽까지 복습합니다.
- **「2. 대표 모델」과 「3. 저장 후 새 Python 프로세스」** → [2일차 슬라이드 23쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s018) · [배포 PDF 23쪽](../슬라이드/2일차.pdf#page=23)에서 30쪽까지 복습합니다.
- **「4. 구조 비교」부터 「완료 확인」까지** → [2일차 슬라이드 31쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s026) · [배포 PDF 31쪽](../슬라이드/2일차.pdf#page=31)에서 35쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 작은 모델의 훈련 전후 예측 또는 Ollama 응답을 저장합니다.
- **단원 마지막 연결 질문:** 직접 훈련한 숫자 과제와 이미 학습된 모델 사용을 구분합니다. 제공한 작업 지침에 없는 질문에는 근거가 없다고 말하는지 확인합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib, json, subprocess, sys, time
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '04.LLM/common').is_dir())
COURSE = ROOT / '04.LLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import DATA, OUTPUTS
from common.contracts import check
from common.loading import load_op
import torch
OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), '훈련은 Jetson GPU에서 수행합니다. 강사에게 환경 점검을 요청하세요.'
def run_training(kind, variant='exercise'):
    before = {p.resolve() for p in OUTPUTS.glob('*/metrics.json')}
    command = [sys.executable, str(COURSE / 'scripts/train.py'), '--model', kind, '--variant', variant]
    result = subprocess.run(command, text=True, capture_output=True, timeout=300)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    created = [p for p in OUTPUTS.glob('*/metrics.json') if p.resolve() not in before]
    assert len(created) == 1, f'새 metrics.json이 정확히 1개여야 합니다: {created}'
    metrics_path = created[0]
    report = json.loads(metrics_path.read_text())
    assert report['kind'] == kind and report['variant'] == variant and report['profile'] == 'quick'
    report['_metrics_path'] = str(metrics_path.relative_to(DATA))
    print('검증 loss:', report['before']['loss'], '→', report['after']['loss'])
    if kind in ('rnn','lstm'):
        print('token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    else:
        print('자유 생성 token 정확도:', report['before']['token_accuracy'], '→', report['after']['token_accuracy'])
        print('자유 생성 시퀀스 전체 일치율:', report['after']['sequence_accuracy'])
    print('실행 경로:', report['run_path'])
    return report


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 1. 숫자 다섯 개를 기억하는 작은 과제

사람이 읽는 입력 `[1,2,3,4,5]`에 대한 기대 출력은 `[BOS,1,2,3,4]`입니다. 첫 위치는 시작 기호, 이후는 직전 입력입니다. 내부에서는 숫자에 3을 더한 token ID를 사용합니다. BOS는 1입니다. 모델이 같은 입력을 그대로 복사하는 과제와 다릅니다.

`B`는 예제 개수, `T`는 순서 길이, `V`는 고를 수 있는 token 종류 수입니다. logits `[B,T,V]`는 각 위치의 후보 점수이고 정답 `[B,T]`는 각 위치에서 맞혀야 하는 번호입니다. 아래에서는 두 예제·세 위치·네 후보만 사용해 축을 봅니다.

Loss는 틀린 정도를 숫자로 요약합니다. backward는 각 파라미터가 loss 변화에 미치는 기울기를 계산하고, optimizer.step은 그 정보를 이용해 파라미터를 수정합니다. 미분식을 직접 유도하지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
example_logits = torch.zeros(2, 3, 4, device='cuda')
example_targets = torch.tensor([[0, 1, 2], [1, 2, 3]], device='cuda')
print('점수:', tuple(example_logits.shape), '정답:', tuple(example_targets.shape))
print('위치별 후보를 모으면:', tuple(example_logits.reshape(-1, 4).shape))
print('정답을 같은 순서로 모으면:', tuple(example_targets.reshape(-1).shape))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 셀 4–6: CUDA와 logits/targets shape 확인; loss TODO 앞에서 멈춤

[2일차 슬라이드 10쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s005) · [배포 PDF 10쪽](../슬라이드/2일차.pdf#page=10)로 돌아가 **10–17쪽** 에서 방금 수행한 내용과 결과를 설명합니다.

복습 후 이 노트북의 **「수업 핵심 TODO — loss」** 부터 이어갑니다.
<!-- slide-sync:end -->

## 수업 핵심 TODO — loss

수정 파일: **[04.LLM/exercises/loss.py](exercises/loss.py)**

[04.LLM/exercises/loss.py](exercises/loss.py)의 한 함수를 완성합니다. 준비된 import·함수 이름은 유지하고 다음 세 연산을 연결합니다.

1. logits의 마지막 축 크기를 후보 수로 읽습니다.
2. logits를 `reshape(-1, logits.shape[-1])`, targets를 `reshape(-1)`로 펼칩니다.
3. `F.cross_entropy`에 두 값을 전달해 반환합니다. softmax는 미리 적용하지 않습니다.

막히면 `[2,3,4]→[6,4]`, `[2,3]→[6]`을 손으로 그린 뒤 검증을 재실행합니다. 구현이 끝나지 않은 경우 강사 도움을 받아 loss 한 함수를 복구하고 진행합니다. 다른 모델 TODO는 선행 조건이 아닙니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `token_loss(logits, targets)`에서 logits는 실수 텐서 `[B,T,V]`, targets는 정답 class 번호를 담은 정수 텐서 `[B,T]`입니다. targets는 같은 device에 있고 값은 `0..V-1`입니다. 반환은 평균 손실을 담은 0차원 텐서이며 역전파 연결을 유지합니다. `.item()`으로 숫자로 바꾸어 반환하지 않습니다.
**작성 흐름 힌트:** 위치 B×T개와 후보 V개를 구분 → 두 입력의 위치 순서를 같이 펼침 → raw logits와 정답 번호를 cross entropy에 전달. 모든 후보 점수가 같은 V=4 예에서는 손실이 약 `log(4)=1.3863`입니다. 이 수치를 고정 반환하는 문제가 아닙니다.
**설명할 지점:** “출력값 하나뿐 아니라 gradient가 모델까지 돌아가야 학습입니다. 먼저 softmax하거나 detach하면 어떤 차이가 생길까요?”
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
print((COURSE / 'exercises/loss.py').read_text())

In [ ]:
# 구분: 수업 핵심 셀
check('loss', 'exercise')
print('learner_verification=PASS')

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다. 수업 중에는 시간이 남을 때 읽습니다.

### 참고 정답

자습에서는 필요할 때 정답과 해설을 열어 비교하고 자신의 코드로 재검증합니다. 아래 실행은 자신의 TODO를 변경하지 않습니다.

In [ ]:
# 구분: 수업 후 복습·자습 셀
print((COURSE / 'solutions/loss.py').read_text())

### 정답 해설 — 입력부터 결과까지

**두 tensor를 같은 순서로 펴기:** logits `[B,T,V]`에서 마지막 축 V는 후보 점수이므로 남기고 앞의 B·T를 합칩니다. targets `[B,T]`도 같은 순서로 펴야 각 점수 행과 정답 번호가 대응합니다. 예를 들어 `[2,3,4]`와 `[2,3]`은 `[6,4]`와 `[6]`입니다.

`F.cross_entropy`에는 확률이 아닌 raw logits와 정수 정답 index를 넣습니다. softmax를 미리 적용하지 않는 이유는 이 함수가 필요한 확률 계산을 내부에서 처리하기 때문입니다. 모든 후보 점수가 같고 후보가 4개라면 정답 확률은 1/4, loss는 약 1.386입니다. 정답 후보 점수가 상대적으로 커지면 loss가 줄어듭니다.

**결과 해석:** loss는 훈련에서 바꿀 방향을 계산할 기준이고, 실제로 파라미터를 갱신하는 것은 backward 뒤 optimizer.step입니다. 낮은 훈련 loss만으로 새로운 입력의 기억 성능을 입증하지 않습니다. 같은 검증 입력의 token/sequence 정확도와 훈련 전후 예측을 같이 봅니다.

**확인 질문:** logits만 다른 순서로 펼치면 shape가 맞아도 왜 엉뚱한 정답으로 학습하게 되나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/loss.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
import torch.nn.functional as F

def token_loss(logits, targets):
    # CrossEntropyLoss는 softmax를 내부에서 처리하므로 raw logits을 전달합니다.
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
check('loss', 'solution')
print('solution_verification=PASS')

> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 셀 7–13: loss.py 저장·학습자 검증; 「2. 대표 모델」 앞에서 멈춤

[2일차 슬라이드 18쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s013) · [배포 PDF 18쪽](../슬라이드/2일차.pdf#page=18)로 돌아가 **18–22쪽** 에서 방금 수행한 내용과 결과를 설명합니다.

복습 후 이 노트북의 **「2. 대표 모델」과 「3. 저장 후 새 Python 프로세스」** 부터 이어갑니다.
<!-- slide-sync:end -->

## 2. 대표 모델을 직접 훈련합니다

<!-- procedure:lesson-10-afe7cd:start -->
![훈련 결과를 저장하고 다시 쓰기](../교재그림/절차/lesson-10.svg)

현재 메모리의 모델과 저장 파일은 다릅니다. 새 프로세스 재로드까지 확인해야 재사용을 검증합니다.
<!-- procedure:lesson-10-afe7cd:end -->


고정 데이터 640개 중 훈련 512개는 파라미터 수정에, 검증 128개는 결과 관찰에 사용합니다. 이번 실습에는 별도의 최종 테스트 집합이 없으므로 현장 성능을 결론내리지 않습니다.

LSTM은 이전 입력에서 가져온 상태를 다음 계산에 사용합니다. 자세한 gate 수식은 자습입니다. 훈련 전후를 비교하되 token 정확도는 위치별 일치, sequence 정확도는 전체 순서 일치라는 차이를 읽으세요.

### 훈련 설정과 loss 기록을 현재 노트북에서 읽기

2일차 슬라이드 「Batch·Epoch·학습률」·「Loss 그래프」에 대응하는 내용입니다. batch는 한 번에 처리하는 예제 묶음, step은 한 번의 파라미터 업데이트, epoch는 전체 훈련 데이터를 한 번 사용하는 분량입니다. `configs/quick.json`에 batch·학습률·LSTM step이 있으며 같은 노트북의 아래 훈련 셀이 이 설정으로 실행합니다. `loss.csv`의 step별 loss를 읽어 대체로 감소하는지, 갑자기 커지는 구간이 있는지 보고 최종 검증 loss·정확도와 함께 해석합니다. 훈련 loss만 낮아지는 것으로 현장 성능을 결론내리지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
core_report = run_training('lstm')
print(json.dumps(core_report['after']['examples'], indent=2))
print('quick 설정:', json.loads((COURSE / 'configs/quick.json').read_text()))
loss_path = DATA / core_report['run_path'] / 'loss.csv'
import csv
with loss_path.open(newline='') as stream:
    loss_rows = list(csv.DictReader(stream))
print('loss 기록:', loss_path, '기록 행:', len(loss_rows))
print('처음:', loss_rows[:3], '마지막:', loss_rows[-3:])


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 3. 저장 후 새 Python 프로세스에서 재사용합니다

메모리의 모델과 디스크의 checkpoint는 다릅니다. 새 프로세스가 가중치를 읽어 같은 규칙을 예측하는지 확인합니다. 이 checkpoint는 추론 재현용이고 optimizer 상태까지 복원하는 훈련 재개용은 아닙니다.


In [ ]:
# 구분: 수업 핵심 셀
checkpoint = DATA / core_report['checkpoint']
assert checkpoint.is_file()
result = subprocess.run([sys.executable, str(COURSE/'scripts/infer.py'),
    '--checkpoint', str(checkpoint), '--digits', '1,2,3,4,5', '--variant', 'exercise'],
    capture_output=True, text=True, timeout=120)
print(result.stdout, result.stderr)
result.check_returncode()
print('기대 출력: BOS, 1, 2, 3, 4. 실제 예측과 비교하세요.')


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 셀 14–17: 훈련 전후 지표·checkpoint 재로드 확인

[2일차 슬라이드 23쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s018) · [배포 PDF 23쪽](../슬라이드/2일차.pdf#page=23)로 돌아가 **23–30쪽** 에서 방금 수행한 내용과 결과를 설명합니다.

복습 후 이 노트북의 **「4. 구조 비교」부터 「완료 확인」까지** 부터 이어갑니다.
<!-- slide-sync:end -->

## 4. 구조 비교는 같은 과제끼리 합니다

[훈련 결과 비교 예시](README.md#훈련-결과-비교-예시)의 RNN과 LSTM 결과로 훈련 시간·같은 과제의 정확도를 비교합니다. 파라미터 수는 모델 요약 출력으로 확인합니다. 다른 역순 과제인 Seq2Seq/Attention/Transformer와 정확도 순위를 만들지 않습니다. 여러 seed와 장비 부하를 측정하지 않은 값은 우열의 일반적 증거가 아닙니다.

강사 시연: RNN의 반복 상태 → LSTM의 기억 상태 → Attention이 입력 위치를 참고하는 방식 → Transformer의 위치 정보. 구현은 각각 01–04에서 자습합니다.

## 5. 업무 판단과 기록

공통 작업대 사례의 이미지 판단에 이 숫자 모델을 바로 쓸 수 없는 이유를 설명하세요. 이미지 데이터·정답 정의·평가 과제가 다릅니다. 기존 모델을 사용해 목표 조건을 먼저 평가하고, 부족한 원인이 데이터·입력 조건·모델 중 무엇인지 구분한 뒤 추가 훈련 여부를 검토합니다.

아래 세 질문에 직접 답한 뒤 저장합니다. 빈 답은 완료가 아닙니다.

### README 비교표의 위치와 내용

위 [README의 「훈련 결과 비교 예시」](README.md#훈련-결과-비교-예시)에는 2026-09-06 Jetson Orin, Python 3.10.12·PyTorch 2.11.0·CUDA 12.6, seed 2026, quick, train 512/validation 128의 다음 **참고 측정값** 이 있습니다. 현재 수강생 측정은 `core_report`이며 아래 표를 자신의 측정값으로 옮기지 않습니다.

| 모델 | step | 훈련 루프 시간 | 검증 loss 전→후 | token 정확도 | 전체 시퀀스 정확도 |
|---|---:|---:|---|---:|---:|
| RNN | 150 | 1.26초 | 2.5786→0.0055 | 100% | 100% |
| LSTM | 200 | 1.69초 | 2.5687→0.0064 | 100% | 100% |

시간에는 프로세스 시작·모델 생성·평가·저장이 포함되지 않고 step 수도 다릅니다. 동일한 짧은 기억 과제의 참고 결과이며 모델의 일반적 우열을 의미하지 않습니다.

**기록 TODO의 입력·출력:** 바로 아래 `training_review`의 빈 문자열 세 곳을 자기 문장으로 채웁니다. 첫 항목에는 `core_report['before']`와 `['after']`에서 읽은 loss·정확도 또는 예측의 실제 변화를, 둘째에는 긴 시퀀스·자연어·현장 성능 등 현재 평가가 확인하지 않은 범위를, 셋째에는 작업대 이미지·판정 라벨·조건별 검증 등 추가로 필요한 데이터를 적습니다. 기록 셀은 `/workspace/Data/outputs/course-review/04.LLM/training-review.json`에 측정 경로와 답을 저장합니다. 세 답이 모두 비어 있지 않고 파일을 다시 열어 측정값·설명을 확인하면 기록 완료입니다.


In [ ]:
# 구분: 수업 핵심 셀
training_review = {
    '내가_관찰한_훈련_전후_차이': '',
    '현재_검증으로_알수없는_것': '',
    '작업대_과제에_필요한_추가_데이터와_판단': '',
}


In [ ]:
# 구분: 수업 핵심 셀
review_path = DATA/'outputs/course-review/04.LLM/training-review.json'
review_path.parent.mkdir(parents=True, exist_ok=True)
review_path.write_text(json.dumps({
    'completed': all(v.strip() for v in training_review.values()),
    'metrics_path': core_report['_metrics_path'],
    'checkpoint': core_report['checkpoint'], 'answers': training_review,
}, ensure_ascii=False, indent=2) + '\n')
print(review_path, '완료:', all(v.strip() for v in training_review.values()))

saved_review = json.loads(review_path.read_text())
assert saved_review['answers'] == training_review
print(json.dumps(saved_review['answers'], ensure_ascii=False, indent=2))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 완료 확인

- 자신의 loss TODO 검사, LSTM quick 훈련, checkpoint 재로드를 완료했습니다.
- 훈련 전후 예측과 지표를 읽고 개선되지 않은 경우도 그대로 기록했습니다.
- 판단 질문 세 개에 답하고 결과 파일을 다시 열었습니다. 코드 실행만으로 서술 평가가 완료되는 것은 아닙니다.
- 훈련 프로세스가 끝난 뒤 05_ollama_basics → 06_ollama_python의 2시간으로 이동합니다. 01–04는 자습입니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 셀 18–21: 비교 결과·질문 3개·기록 저장; 모델 5개 추가 훈련 없음

[2일차 슬라이드 31쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s026) · [배포 PDF 31쪽](../슬라이드/2일차.pdf#page=31)로 돌아가 **31–35쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

